# Амортизация, шаг 2: сверка расчёта DAG с таблицей `..._jan_aug` за август 2026

Сначала запустите `amort_01_price_ref_load.ipynb` (справочник цен в озере).

Здесь тот же SQL, что в `include/sql/acq_04a_amortization.sql`, только окно — один август и без `CREATE TABLE`. Результат сравнивается с `sandbox_ai.shestopalov_terminal_amortization_model_jan_aug` по каждому терминалу.

Только чтение. Логин keytab спрашивается один раз.

In [ ]:
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

PRICE_TABLE = "sandbox_ai.shestopalov_terminal_model_price"
OLD_AMORT_TABLE = "sandbox_ai.shestopalov_terminal_amortization_model_jan_aug"
MONTH_START, MONTH_END, REPORT_MONTH = "2026-08-01", "2026-08-31", "2026-08"

if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def imp_fetch(sql):
    with imp:
        imp.execute("SET MEM_LIMIT=8g")
        return imp.fetch(sql)

## 1. Расчёт как в DAG за август

In [ ]:
MONTHS_SUBQUERY = (
    f"(select cast('{MONTH_START}' as date) as month_start, cast('{MONTH_END}' as date) as month_end, "
    f"'{REPORT_MONTH}' as report_month)"
)

SQL_DAG = """
with terminals as (
  select
    m.month_start,
    m.report_month,
    cast(t.c_nter as string) as c_nter,
    cast(t.c_pos_serial as string) as c_pos_serial,
    row_number() over (
      partition by m.month_start, cast(t.c_nter as string)
      order by
        coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) desc,
        coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) desc,
        cast(t.c_pos_serial as string) desc
    ) as rn
  from ods_alpha.scd1_pos_terminals t
  join __MONTHS__ m
    on coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= m.month_end
   and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= m.month_start
  where t.c_nter is not null
    and nullif(trim(cast(t.c_pos_serial as string)), '') is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
),
first_delivery as (
  select
    cast(c_pos_serial as string) as c_pos_serial,
    min(cast(d_ter_delivery as date)) as first_d_ter_delivery
  from ods_alpha.scd1_pos_terminals
  where c_pos_serial is not null
    and d_ter_delivery is not null
    and coalesce(ods_deleted_flg, '0') <> '1'
  group by cast(c_pos_serial as string)
),
device as (
  select
    case when trim(cast(id_device as string)) rlike '^[0-9]+$'
         then cast(cast(trim(cast(id_device as string)) as bigint) as string)
         else upper(trim(cast(id_device as string))) end as id_norm,
    case when trim(cast(code_device as string)) rlike '^[0-9]+$'
         then cast(cast(trim(cast(code_device as string)) as bigint) as string)
         else upper(trim(cast(code_device as string))) end as code_norm,
    trim(cast(model_device as string)) as model_device
  from dwh.bmrt_bm_det_device
  where nullif(trim(cast(model_device as string)), '') is not null
    and lower(trim(cast(model_device as string))) not in ('none', 'nan', 'null', '<null>', 'nat')
),
by_id as (
  select id_norm, max(model_device) as model_device from device where id_norm is not null group by id_norm
),
by_code as (
  select code_norm, max(model_device) as model_device from device where code_norm is not null group by code_norm
),
term_model as (
  select
    t.month_start,
    t.report_month,
    t.c_nter,
    t.c_pos_serial,
    fd.first_d_ter_delivery,
    coalesce(i.model_device, c.model_device) as model_device,
    case when i.model_device is not null then 'id_device'
         when c.model_device is not null then 'code_device' end as join_key_used
  from terminals t
  left join first_delivery fd
    on fd.c_pos_serial = t.c_pos_serial
  left join by_id i
    on i.id_norm = case when trim(t.c_nter) rlike '^[0-9]+$'
                        then cast(cast(trim(t.c_nter) as bigint) as string)
                        else upper(trim(t.c_nter)) end
  left join by_code c
    on c.code_norm = case when trim(t.c_nter) rlike '^[0-9]+$'
                          then cast(cast(trim(t.c_nter) as bigint) as string)
                          else upper(trim(t.c_nter)) end
  where t.rn = 1
),
priced as (
  select
    tm.*,
    p.price,
    coalesce(p.is_excluded, 0) as is_excluded,
    case when tm.first_d_ter_delivery is not null
         then (year(tm.month_start) - year(tm.first_d_ter_delivery)) * 12
              + (month(tm.month_start) - month(tm.first_d_ter_delivery)) end as months_from_start
  from term_model tm
  left join __PRICE__ p
    on p.model_key = trim(regexp_replace(lower(replace(tm.model_device, 'ё', 'е')), '[[:space:]]+', ' '))
)
select
  report_month,
  month_start as snapshot_month_start,
  c_nter,
  c_pos_serial,
  first_d_ter_delivery,
  model_device,
  join_key_used,
  price,
  is_excluded,
  months_from_start,
  case
    when first_d_ter_delivery is null then 'missing_deliver'
    when model_device is null then 'missing_model'
    when price is null then 'missing_price'
    else 'complete'
  end as qc_status,
  case
    when is_excluded = 1 or price is null then 0.0
    when months_from_start >= 0 and months_from_start < 48 then cast(price as double) / 48.0
    else 0.0
  end as amortization_for_report_month
from priced
""".replace("__MONTHS__", MONTHS_SUBQUERY).replace("__PRICE__", PRICE_TABLE)

with imp:
    imp.execute("REFRESH dwh.bmrt_bm_det_device")
    imp.execute(f"REFRESH {PRICE_TABLE}")
new = imp_fetch(SQL_DAG)
new["c_nter"] = new["c_nter"].astype(str).str.strip()
new["new_amort"] = pd.to_numeric(new["amortization_for_report_month"], errors="coerce").fillna(0)
print("Терминалов в расчёте DAG:", len(new), "| дублей c_nter:", int(new["c_nter"].duplicated().sum()))
display(
    new.groupby("qc_status", as_index=False)
    .agg(terminals=("c_nter", "size"), amort=("new_amort", "sum"))
    .sort_values("terminals", ascending=False)
)

## 2. Сверка со старой таблицей по терминалу

Хорошо, если `match_pct` около 99–100% и суммы почти равны. Расхождения смотрим по парам моделей ниже.

In [ ]:
old = imp_fetch(f"""
    SELECT cast(c_nter AS string) AS c_nter,
           cast(model_device AS string) AS old_model,
           cast(amortization_for_report_month AS double) AS old_amort
    FROM {OLD_AMORT_TABLE}
    WHERE cast(snapshot_month_start AS string) = '{MONTH_START}'
""")
old["c_nter"] = old["c_nter"].astype(str).str.strip()
old["old_amort"] = pd.to_numeric(old["old_amort"], errors="coerce").fillna(0)

cmp = old.merge(new[["c_nter", "model_device", "qc_status", "new_amort"]], on="c_nter", how="outer", indicator=True)
cmp[["old_amort", "new_amort"]] = cmp[["old_amort", "new_amort"]].fillna(0)
both = cmp["_merge"] == "both"
same = both & ((cmp["old_amort"] - cmp["new_amort"]).abs() < 0.01)

summary = pd.DataFrame([
    {"check": "терминалов в старой", "value": int((cmp["_merge"] != "right_only").sum())},
    {"check": "терминалов в DAG", "value": int((cmp["_merge"] != "left_only").sum())},
    {"check": "только в старой", "value": int((cmp["_merge"] == "left_only").sum())},
    {"check": "только в DAG", "value": int((cmp["_merge"] == "right_only").sum())},
    {"check": "match_pct (общие терминалы)", "value": round(100 * same.sum() / max(both.sum(), 1), 2)},
    {"check": "сумма: старая", "value": round(cmp["old_amort"].sum(), 2)},
    {"check": "сумма: DAG", "value": round(cmp["new_amort"].sum(), 2)},
])
display(summary)

print("Топ-20 расхождений по парам моделей:")
diff = cmp[both & ~same].assign(delta=lambda d: d["new_amort"] - d["old_amort"])
display(
    diff.groupby(["old_model", "model_device", "qc_status"], dropna=False, as_index=False)
    .agg(terminals=("c_nter", "size"), old_amort=("old_amort", "max"), new_amort=("new_amort", "max"),
         delta_sum=("delta", "sum"))
    .sort_values("delta_sum", key=lambda s: s.abs(), ascending=False)
    .head(20)
)

print("Модели без цены в справочнике (топ-20 по числу терминалов):")
display(
    new[new["qc_status"] == "missing_price"]
    .groupby("model_device", as_index=False).size()
    .sort_values("size", ascending=False).head(20)
)